# 01 · Bronze layer — ingestion of MovieLens 25M into DuckLake

**Pipeline:** `01_bronze` → `02_silver` → `03_gold` → `04_analysis`

This notebook:

1. Profiles the downloaded source files (evidence for Task 1: files, formats, rows, columns).
2. Simulates two source deliveries by splitting `ratings.csv` on its `timestamp` watermark
   (batch 1 = ratings before 2015, batch 2 = 2015 onward), so the incremental design from Task 2 can be demonstrated.
3. Loads the files into the Bronze schema **as raw text, with no cleaning**, adding tracking metadata
   (`source_name`, `source_file`, `load_id`, `ingested_at`).
4. Records every load in `bronze.load_log` with the input row count, loaded row count and the batch's watermark range.

Design rationale for the ingestion strategy (full vs incremental, batch vs streaming, push vs pull,
frequency, recovery) is in `README.md`, section *Task 2*.

In [1]:
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

from lakehouse import LANDING_DIR, RAW_DIR, connect, find_raw_file, sql_path

pd.set_option("display.max_colwidth", 80)

# Split point used to simulate a second, later delivery of ratings.
BATCH_CUTOFF_UTC = "2015-01-01"

con = connect()
print("DuckDB", con.sql("SELECT version()").fetchone()[0], "| lakehouse attached as 'lake'")

DuckDB v1.5.6 | lakehouse attached as 'lake'


## 1. Source profile (Task 1 evidence)

Counts every row and column of each CSV in the download. All columns are read as text so the profile
reflects the files exactly as delivered.

In [2]:
profile = []
for f in sorted(RAW_DIR.rglob("*.csv")):
    src = f"read_csv({sql_path(f)}, header = true, all_varchar = true)"
    n_rows = con.sql(f"SELECT count(*) FROM {src}").fetchone()[0]
    cols = con.sql(f"DESCRIBE SELECT * FROM {src}").df()["column_name"].tolist()
    profile.append({
        "file": f.name,
        "format": "CSV (comma-separated, header row)",
        "size_MB": round(f.stat().st_size / 1e6, 1),
        "rows": n_rows,
        "columns": len(cols),
        "column_names": ", ".join(cols),
    })
profile = pd.DataFrame(profile)
print(f"{len(profile)} CSV files found under {RAW_DIR}/")
display(profile)

6 CSV files found under data\raw/


,file,format,size_MB,rows,columns,column_names
0,genome-scores.csv,"CSV (comma-separated, header row)",435.2,15584448,3,"movieId, tagId, relevance"
1,genome-tags.csv,"CSV (comma-separated, header row)",0.0,1128,2,"tagId, tag"
2,links.csv,"CSV (comma-separated, header row)",1.4,62423,3,"movieId, imdbId, tmdbId"
3,movies.csv,"CSV (comma-separated, header row)",3.0,62423,3,"movieId, title, genres"
4,ratings.csv,"CSV (comma-separated, header row)",678.3,25000095,4,"userId, movieId, rating, timestamp"
5,tags.csv,"CSV (comma-separated, header row)",38.8,1093360,4,"userId, movieId, tag, timestamp"


Only `ratings.csv` and `movies.csv` are needed for the four questions, so only those two enter the lakehouse.
`tags.csv`, `links.csv`, `genome-scores.csv` and `genome-tags.csv` are deliberately **not** ingested
(they do not support any of the questions; ingesting them would add ~16M rows of unused data).

## 2. Simulate two source deliveries (landing zone)

The real dataset is one snapshot. To demonstrate the *full-first-then-incremental* design, the pull job's
landing zone receives:

| Delivery | Files | Load type |
|---|---|---|
| 1 | `movies.csv` (full catalogue snapshot) + `ratings_1995_2014.csv` | initial full load |
| 2 | `ratings_2015_2019.csv` | incremental (rows newer than the watermark) |

Rows are split on the raw `timestamp` column. A row whose timestamp is not a valid integer is kept in batch 1
(not dropped) so that Silver can quarantine it — Bronze never silently discards data.
Files are written to a `.part` name first and renamed when complete, so an interrupted run never leaves a
half-written file that looks finished.

In [3]:
LANDING_DIR.mkdir(parents=True, exist_ok=True)

ratings_src = find_raw_file("ratings.csv")
movies_src = find_raw_file("movies.csv")
cutoff_epoch = int(datetime.fromisoformat(BATCH_CUTOFF_UTC).replace(tzinfo=timezone.utc).timestamp())

landing = {
    "movies": LANDING_DIR / "movies.csv",
    "ratings_batch1": LANDING_DIR / "ratings_1995_2014.csv",
    "ratings_batch2": LANDING_DIR / "ratings_2015_2019.csv",
}
raw_ratings = f"read_csv({sql_path(ratings_src)}, header = true, all_varchar = true)"
splits = {
    "ratings_batch1": f'coalesce(TRY_CAST("timestamp" AS BIGINT) < {cutoff_epoch}, true)',
    "ratings_batch2": f'TRY_CAST("timestamp" AS BIGINT) >= {cutoff_epoch}',
}

def write_atomically(target, select_sql):
    part = target.with_name(target.name + ".part")
    con.execute(f"COPY ({select_sql}) TO {sql_path(part)} (FORMAT csv, HEADER true)")
    part.replace(target)

for key, condition in splits.items():
    if landing[key].exists():
        print(f"{landing[key].name}: already in landing zone, not regenerated")
    else:
        write_atomically(landing[key], f"SELECT * FROM {raw_ratings} WHERE {condition}")
        print(f"{landing[key].name}: written")

if landing["movies"].exists():
    print("movies.csv: already in landing zone, not regenerated")
else:
    write_atomically(landing["movies"],
                     f"SELECT * FROM read_csv({sql_path(movies_src)}, header = true, all_varchar = true)")
    print("movies.csv: written")

# The two batches must add up to the original file: nothing lost, nothing duplicated.
def csv_rows(p):
    return con.sql(f"SELECT count(*) FROM read_csv({sql_path(p)}, header = true, all_varchar = true)").fetchone()[0]

n_src = csv_rows(ratings_src)
n_b1, n_b2 = csv_rows(landing["ratings_batch1"]), csv_rows(landing["ratings_batch2"])
display(pd.DataFrame({
    "file": [ratings_src.name, landing["ratings_batch1"].name, landing["ratings_batch2"].name],
    "rows": [n_src, n_b1, n_b2],
}))
assert n_b1 + n_b2 == n_src, "split lost or duplicated rows"
print(f"Split check passed: {n_b1:,} + {n_b2:,} = {n_src:,}")

ratings_1995_2014.csv: written


ratings_2015_2019.csv: written
movies.csv: written


,file,rows
0,ratings.csv,25000095
1,ratings_1995_2014.csv,17436354
2,ratings_2015_2019.csv,7563741


Split check passed: 17,436,354 + 7,563,741 = 25,000,095


## 3. Bronze tables

Bronze stores every source column as `VARCHAR`, exactly as delivered — no trimming, casting, filtering or
de-duplication (that is Silver's job). Four metadata columns are appended to every row for traceability:

| Column | Meaning |
|---|---|
| `source_name` | logical source (`movielens_ratings`, `movielens_movies`) |
| `source_file` | the landing file the row came from |
| `load_id` | the load run that inserted the row (`load_0001`, `load_0002`, …) |
| `ingested_at` | UTC time the load ran |

In [4]:
con.execute('''
CREATE TABLE IF NOT EXISTS bronze.ratings_raw (
    userId VARCHAR, movieId VARCHAR, rating VARCHAR, "timestamp" VARCHAR,
    source_name VARCHAR, source_file VARCHAR, load_id VARCHAR, ingested_at TIMESTAMP
)''')
con.execute('''
CREATE TABLE IF NOT EXISTS bronze.movies_raw (
    movieId VARCHAR, title VARCHAR, genres VARCHAR,
    source_name VARCHAR, source_file VARCHAR, load_id VARCHAR, ingested_at TIMESTAMP
)''')
con.execute('''
CREATE TABLE IF NOT EXISTS bronze.load_log (
    load_id VARCHAR, source_name VARCHAR, source_file VARCHAR, file_size_bytes BIGINT,
    load_type VARCHAR, input_row_count BIGINT, loaded_row_count BIGINT,
    watermark_column VARCHAR, batch_min_watermark TIMESTAMP, batch_max_watermark TIMESTAMP,
    ingested_at TIMESTAMP, status VARCHAR, message VARCHAR
)''')
print("Bronze tables ready:", con.sql(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_catalog = 'lake' AND table_schema = 'bronze' ORDER BY 1"
).df()["table_name"].tolist())

Bronze tables ready: ['load_log', 'movies_raw', 'ratings_raw']


## 4. Load function — idempotent, transactional, logged

`ingest_file` is the whole ingestion contract:

* **Idempotent:** a file already logged as `SUCCESS` (same name and size) is skipped, so rerunning this
  notebook never double-loads data.
* **Atomic:** the insert and its log entry are one transaction. If anything fails (e.g. the source schema
  changed and a column is missing), the transaction rolls back, a `FAILED` row is logged, and the file is
  simply retried on the next run.
* **Counted:** input rows are counted from the file and compared with rows actually inserted.
* **Watermarked:** for ratings, the min/max `timestamp` of the batch is recorded; the max of all successful
  loads is the current high-water mark.

In [5]:
def next_load_id():
    n = con.sql("SELECT count(DISTINCT load_id) FROM bronze.load_log").fetchone()[0]
    return f"load_{n + 1:04d}"


def current_watermark():
    return con.sql('''
        SELECT max(batch_max_watermark) FROM bronze.load_log
        WHERE source_name = 'movielens_ratings' AND status = 'SUCCESS'
    ''').fetchone()[0]


def ingest_file(load_id, source_name, path, target, columns, load_type, watermark_col=None):
    size = path.stat().st_size
    already = con.execute('''
        SELECT count(*) FROM bronze.load_log
        WHERE source_file = ? AND file_size_bytes = ? AND status = 'SUCCESS'
    ''', [path.name, size]).fetchone()[0]
    if already:
        print(f"  skip {path.name}: already loaded successfully")
        return False

    src = f"read_csv({sql_path(path)}, header = true, all_varchar = true)"
    input_rows = con.sql(f"SELECT count(*) FROM {src}").fetchone()[0]
    ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
    col_list = ", ".join(f'"{c}"' for c in columns)

    con.execute("BEGIN TRANSACTION")
    try:
        con.execute(
            f"INSERT INTO {target} SELECT {col_list}, ?, ?, ?, ? FROM {src}",
            [source_name, path.name, load_id, ingested_at],
        )
        loaded_rows = con.execute(
            f"SELECT count(*) FROM {target} WHERE load_id = ? AND source_file = ?",
            [load_id, path.name],
        ).fetchone()[0]
        if loaded_rows != input_rows:
            raise RuntimeError(f"row count mismatch: file {input_rows:,} vs loaded {loaded_rows:,}")

        wm_min = wm_max = None
        if watermark_col:
            wm_min, wm_max = con.execute(f'''
                WITH ts AS (
                    SELECT TRY_CAST("{watermark_col}" AS BIGINT) AS e FROM {target}
                    WHERE load_id = ? AND source_file = ?
                )
                -- ignore unparseable / absurd values here; Silver quarantines them
                SELECT epoch_ms(min(e) * 1000), epoch_ms(max(e) * 1000)
                FROM ts WHERE e BETWEEN 0 AND 4102444800
            ''', [load_id, path.name]).fetchone()

        con.execute(
            "INSERT INTO bronze.load_log VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'SUCCESS', NULL)",
            [load_id, source_name, path.name, size, load_type, input_rows, loaded_rows,
             watermark_col, wm_min, wm_max, ingested_at],
        )
        con.execute("COMMIT")
        print(f"  loaded {path.name}: {loaded_rows:,} rows ({load_type})")
        return True
    except Exception as exc:
        con.execute("ROLLBACK")
        con.execute(
            "INSERT INTO bronze.load_log VALUES (?, ?, ?, ?, ?, ?, 0, ?, NULL, NULL, ?, 'FAILED', ?)",
            [load_id, source_name, path.name, size, load_type, input_rows, watermark_col,
             ingested_at, str(exc)[:500]],
        )
        raise


RATINGS_COLS = ["userId", "movieId", "rating", "timestamp"]
MOVIES_COLS = ["movieId", "title", "genres"]

## 5. Load 1 — initial full load (movies snapshot + ratings up to 2014)

In [6]:
load_id = next_load_id()
print(f"Load {load_id}: watermark before load = {current_watermark()}")
ingest_file(load_id, "movielens_movies", landing["movies"], "bronze.movies_raw",
            MOVIES_COLS, load_type="full snapshot")
ingest_file(load_id, "movielens_ratings", landing["ratings_batch1"], "bronze.ratings_raw",
            RATINGS_COLS, load_type="initial full", watermark_col="timestamp")
print(f"Watermark after load 1 = {current_watermark()}")

Load load_0001: watermark before load = None


  loaded movies.csv: 62,423 rows (full snapshot)


  loaded ratings_1995_2014.csv: 17,436,354 rows (initial full)
Watermark after load 1 = 2014-12-31 23:49:53


## 6. Load 2 — incremental load (ratings from 2015 onward)

Before loading, the current high-water mark is read from the load log. The new batch is checked against
it: every row in an incremental batch should be **newer** than the watermark. Rows at or before the
watermark are still loaded (Bronze keeps everything) but counted as *late-arriving*; Silver's
key-based de-duplication (latest timestamp wins per user × movie) makes them safe.

In [7]:
watermark_before = current_watermark()
load_id = next_load_id()
print(f"Load {load_id}: watermark before load = {watermark_before}")
loaded = ingest_file(load_id, "movielens_ratings", landing["ratings_batch2"], "bronze.ratings_raw",
                     RATINGS_COLS, load_type="incremental", watermark_col="timestamp")

if loaded:
    late = con.execute('''
        SELECT count(*) FROM bronze.ratings_raw
        WHERE source_file = ?
          AND TRY_CAST("timestamp" AS BIGINT) BETWEEN 0 AND 4102444800
          AND epoch_ms(TRY_CAST("timestamp" AS BIGINT) * 1000) <= ?
    ''', [landing["ratings_batch2"].name, watermark_before]).fetchone()[0]
    print(f"Late-arriving rows (timestamp <= previous watermark): {late:,}")
print(f"Watermark after load 2 = {current_watermark()}")

Load load_0002: watermark before load = 2014-12-31 23:49:53


  loaded ratings_2015_2019.csv: 7,563,741 rows (incremental)
Late-arriving rows (timestamp <= previous watermark): 0
Watermark after load 2 = 2019-11-21 09:15:03


## 7. Load evidence

The load log is the audit trail: one row per file per load, with input vs loaded counts and the watermark range.

In [8]:
display(con.sql('''
    SELECT load_id, source_name, source_file, load_type, input_row_count, loaded_row_count,
           input_row_count = loaded_row_count AS counts_match,
           batch_min_watermark, batch_max_watermark, ingested_at, status
    FROM bronze.load_log ORDER BY load_id, source_name
''').df())

,load_id,source_name,source_file,load_type,input_row_count,loaded_row_count,counts_match,batch_min_watermark,batch_max_watermark,ingested_at,status
0,load_0001,movielens_movies,movies.csv,full snapshot,62423,62423,True,NaT,NaT,2026-10-14 15:51:06.445789,SUCCESS
1,load_0001,movielens_ratings,ratings_1995_2014.csv,initial full,17436354,17436354,True,1995-01-09 11:46:49,2014-12-31 23:49:53,2026-10-14 15:51:06.746088,SUCCESS
2,load_0002,movielens_ratings,ratings_2015_2019.csv,incremental,7563741,7563741,True,2015-01-01 00:02:15,2019-11-21 09:15:03,2026-10-14 15:51:08.001942,SUCCESS


In [9]:
# Row counts by load as stored in Bronze — must equal the log above.
display(con.sql('''
    SELECT 'ratings_raw' AS bronze_table, load_id, source_file, count(*) AS rows_in_bronze
    FROM bronze.ratings_raw GROUP BY ALL
    UNION ALL
    SELECT 'movies_raw', load_id, source_file, count(*) FROM bronze.movies_raw GROUP BY ALL
    ORDER BY bronze_table, load_id
''').df())

bronze_ratings = con.sql("SELECT count(*) FROM bronze.ratings_raw").fetchone()[0]
assert bronze_ratings == n_src, "Bronze ratings do not reconcile with the source file"
print(f"Reconciliation passed: Bronze holds all {bronze_ratings:,} source ratings rows.")

,bronze_table,load_id,source_file,rows_in_bronze
0,movies_raw,load_0001,movies.csv,62423
1,ratings_raw,load_0001,ratings_1995_2014.csv,17436354
2,ratings_raw,load_0002,ratings_2015_2019.csv,7563741


Reconciliation passed: Bronze holds all 25,000,095 source ratings rows.


### Raw data preserved

A sample of Bronze rows: values are exactly as in the CSV (text), with the four tracking columns appended.

In [10]:
display(con.sql("SELECT * FROM bronze.ratings_raw LIMIT 5").df())
display(con.sql("SELECT * FROM bronze.movies_raw LIMIT 5").df())
display(con.sql("DESCRIBE bronze.ratings_raw").df()[["column_name", "column_type"]])

,userId,movieId,rating,timestamp,source_name,source_file,load_id,ingested_at
0,27958,2822,3.0,1033601298,movielens_ratings,ratings_1995_2014.csv,load_0001,2026-10-14 15:51:06.746088
1,27958,2829,2.0,1033602516,movielens_ratings,ratings_1995_2014.csv,load_0001,2026-10-14 15:51:06.746088
2,27958,2836,4.0,1033602160,movielens_ratings,ratings_1995_2014.csv,load_0001,2026-10-14 15:51:06.746088
3,27958,2841,3.0,1033606308,movielens_ratings,ratings_1995_2014.csv,load_0001,2026-10-14 15:51:06.746088
4,27958,2844,3.0,1033604318,movielens_ratings,ratings_1995_2014.csv,load_0001,2026-10-14 15:51:06.746088


,movieId,title,genres,source_name,source_file,load_id,ingested_at
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,movielens_movies,movies.csv,load_0001,2026-10-14 15:51:06.445789
1,2,Jumanji (1995),Adventure|Children|Fantasy,movielens_movies,movies.csv,load_0001,2026-10-14 15:51:06.445789
2,3,Grumpier Old Men (1995),Comedy|Romance,movielens_movies,movies.csv,load_0001,2026-10-14 15:51:06.445789
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,movielens_movies,movies.csv,load_0001,2026-10-14 15:51:06.445789
4,5,Father of the Bride Part II (1995),Comedy,movielens_movies,movies.csv,load_0001,2026-10-14 15:51:06.445789


,column_name,column_type
0,userId,VARCHAR
1,movieId,VARCHAR
2,rating,VARCHAR
3,timestamp,VARCHAR
4,source_name,VARCHAR
5,source_file,VARCHAR
6,load_id,VARCHAR
7,ingested_at,TIMESTAMP


In [11]:
con.close()  # release the DuckLake catalog so the next notebook can attach it
print("Bronze layer complete.")

Bronze layer complete.